In [ ]:
from pathlib import Path
import os
from dotenv import load_dotenv
from mhc.util import find_root, run_module

load_dotenv(dotenv_path=f"{find_root(Path.cwd())}/.env", override=True)
from mhc.config import PROJECT_DIRECTORY

In [ ]:
experiment_names = [experiment.strip() for experiment in os.getenv("ME_EVALUATION_EXPERIMENTS_NAMES").split(",")]

In [ ]:
run_module(
    "ptc.migrate.copy_experiment_artifacts",
    args=[
        "--source-experiment-name", "main",
        "--destination-experiment-name", "t2plinker",
    ],
)

In [ ]:
for experiment_name in experiment_names:
    run_module("ptc.generator.expand_t2p_candidate",
               args=[f"--experiment-name={experiment_name}"])

In [ ]:
for experiment_name in experiment_names:
    run_module("ptc.generator.filter_t2p_candidate",
               args=[f"--experiment-name={experiment_name}",
                     f"--t2p-ground-truth-dir={PROJECT_DIRECTORY}/data/{experiment_name}/t2p-ground-truth", "--replace"])

In [ ]:
for experiment_name in experiment_names:
    run_module("ptc.testlinker.main", args=[
        "testlinker",
        "--stage=all",
        f"--experiment-name={experiment_name}",
        "--tokenizer-mode=auto",
        "--include-labels",
        "--project-index=:",
        "--method-resolver=all",
        "--order-production-method=testlinker",
        "--model-name-or-path=Salesforce/codet5-base",
        "--replace",
    ])

In [ ]:
for experiment_name in experiment_names:
    run_module("ptc.generator.t2p_technique", args=[
        f"--experiment-name={experiment_name}",
        f"--strategies=all",
        "--replace",
    ])

In [ ]:
for experiment_name in experiment_names:
    run_module("ptc.generator.t2p_link", args=[
        f"--experiment-name={experiment_name}",
        f"--strategies=",
        "--replace",
    ])

In [ ]:
run_module(
    "ptc.generator.t2p_evaluation",
    args=[
        "--ground-truth-config=config/t2p_ground_truth.yml",
        "--experiment-group=all",
    ],
)